# 6 · Species handling: transported H⁺ or not

| `species` | |
|---|---|
| `with_H` (default) | the proton total H_T is transported and consumed or released by the reactions (the original pH-cell line) |
| `no_H` | H_T is held at its initial value; only Zn, Mn and SO₄ move (the original charge line) |

With `no_H` the pH still comes from the speciation, but it can follow only the Zn, Mn and SO₄ totals: the protons the reactions consume are not tracked.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model

def compare(runs, title, cols=("V", "pH_cath")):
    """Plot several runs (dict label -> Result) against capacity."""
    fig, ax = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4.2))
    ax = np.atleast_1d(ax)
    for label, r in runs.items():
        for a, c in zip(ax, cols):
            a.plot(r.column("mAhg"), r.column(c), label=label)
    for a, c in zip(ax, cols):
        a.set(xlabel="capacity [mAh/g]  (decreasing = charge)", ylabel=c)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def summary(runs):
    for label, r in runs.items():
        q = r.column("mAhg")
        print(f"{label:28s} exit {r.exit_reason:16s} discharged {q.max():6.1f} mAh/g, final {q[-1]:6.1f} mAh/g")

In [ ]:
runs = {s: run(Params(species=s, steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False)) for s in ("with_H", "no_H")}
summary(runs)
compare(runs, "species handling", cols=("V", "pH_cath", "vf_ZHS"));

The same with acid in the electrolyte, where the protons matter most:

In [ ]:
runs = {s: run(Params(species=s, c_H2SO4=0.1, steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False)) for s in ("with_H", "no_H")}
summary(runs)
compare(runs, "species handling, 0.1 M H2SO4", cols=("V", "pH_cath"));